# 02 · Fine-tuning SQL con LoRA · la cura del olvido

> Misma tarea que el Lab 1 (GPT-2 -> SQL), mismo dataset y misma evaluación. Pero con LoRA:
> casi la misma accuracy, entrenando una fracción minúscula de los pesos, y SIN olvidar lo que el modelo ya sabia.

## Qué aprendemos

1. **LoRA** : congelar el modelo y entrenar solo unos adapters de bajo rango.
2. El **A/B contra el Lab 1** : accuracy de SQL cercana (en la ejecución de referencia, oct-2026 en T4: ~51% de match exacto con LoRA frente a ~62% con full FT), pero el modelo base queda INTACTO.
3. Los **adapters** : minusculos de guardar y se pueden quitar o intercambiar.

## Por qué importa

El Lab 1 nos dejo dos dolores: el full fine-tuning es **caro** (124M de pesos) y **destruye** el modelo
(olvido catastrofico). LoRA cura los dos a la vez. Es la técnica que se usa hoy.

## Conecta con

- **Slides de Fine-tuning · Deck 2** : "LoRA la transparencia" y "matrices pequeñas".
- **Lab 01** : la misma tarea con full FT, para comparar A/B.
- **Lab 03** : LoRA escala a QLoRA en un modelo moderno (Gemma 4 + Unsloth, routing de intención con banking77).

## Stack

`gpt2` (124M) · `transformers` · `peft` · `datasets` · `b-mc2/sql-create-context`.

## Hardware

T4 (Colab Free). Mismo modelo que el Lab 1, pero entrena muchos menos pesos. Ejecución de referencia (oct-2026): ~8 min en total.


---

## Setup


In [1]:
!pip install -qU transformers datasets accelerate peft torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 110.5 MB/s eta 0:00:00


In [2]:
import math, re, sqlite3, os, torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


---

# 1 · Cargamos GPT-2 y le ponemos LoRA

Cargamos el MISMO GPT-2 del Lab 1. Pero en vez de tocar sus pesos, los **congelamos** y anadimos
adapters de bajo rango (LoRA) en la atención (`c_attn`, `c_proj`) y en el MLP (`c_fc`).


In [3]:
MODEL = "gpt2"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL).to(device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [4]:
lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                       # rango del adapter
    lora_alpha=16,             # escala (= 2r)
    lora_dropout=0.05,
    # Atención (c_attn = Q,K,V juntas; c_proj = salida) + MLP (c_fc). c_proj aparece en atención y en MLP.
    # Con solo c_attn el adapter se queda corto (~30% de match exacto frente al ~60% del full FT);
    # cubriendo también el MLP se acerca al Lab 1 y sigue por debajo del 1% de los pesos.
    target_modules=["c_attn", "c_proj", "c_fc"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()  # fijate en el % minusculo

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2760: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Ese porcentaje es la clave: entrenaremos **menos del 1%** de los pesos, no los 124M del Lab 1.


---

# 2 · El dataset · el mismo del Lab 1


In [5]:
def build_prompt(question, context):
    return (
        "Given the database schema, write the SQL query.\n"
        f"Schema: {context}\n"
        f"Question: {question}\n"
        "SQL:"
    )

ds = load_dataset("b-mc2/sql-create-context", split="train")

# Igual que en el Lab 1: el "\n" tras la query le enseña a parar (el EOS = pad queda enmascarado en las labels)
def to_text(ex):
    return {"text": build_prompt(ex["question"], ex["context"]) + " " + ex["answer"] + "\n" + tok.eos_token}

ds = ds.map(to_text).shuffle(seed=3407)
train_ds = ds.select(range(8000))
eval_ds  = ds.select(range(8000, 8300))

def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=256)

train_tok = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
collator = DataCollatorForLanguageModeling(tok, mlm=False)
print("train:", len(train_ds), "· eval:", len(eval_ds))

train: 8000 · eval: 300


---

# 3 · Helpers de evaluación + el "antes"

Mismas funciones que el Lab 1, para que la comparación sea limpia. El adapter recien creado tiene
B=0, así que de momento el modelo se comporta EXACTAMENTE como GPT-2 base.


In [6]:
@torch.no_grad()
def generate(prompt, max_new_tokens=48):
    model.eval()
    enc = tok(prompt, return_tensors="pt").to(device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.split("\n")[0].strip()

@torch.no_grad()
def general_completion(prompt, max_new_tokens=30):
    model.eval()
    enc = tok(prompt, return_tensors="pt").to(device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0], skip_special_tokens=True)

def normalize_sql(s):
    s = s.strip().rstrip(";").lower()
    return re.sub(r"\s+", " ", s)

@torch.no_grad()
def sql_accuracy(n=120, show=5):
    ok = 0
    for i, ex in enumerate(eval_ds.select(range(n))):
        pred = generate(build_prompt(ex["question"], ex["context"]))
        match = normalize_sql(pred) == normalize_sql(ex["answer"])
        ok += int(match)
        if i < show:   # vemos en directo los SQL generados
            print(f"[{'OK ' if match else 'FALLO'}] {ex['question']}")
            print(f"   generado: {pred}")
            print(f"   esperado: {ex['answer']}\n")
    return ok / n

GENERAL_TEXT = (
    "The history of artificial intelligence began in antiquity, with myths and stories "
    "of artificial beings endowed with intelligence by master craftsmen."
)
GENERAL_PROMPTS = [
    "Once upon a time, in a small village,",
    "The capital of France is",
    "To make a good cup of coffee, you need",
]

@torch.no_grad()
def perplexity(text):
    model.eval()
    enc = tok(text, return_tensors="pt").to(device)
    loss = model(**enc, labels=enc["input_ids"]).loss
    return math.exp(loss.item())

# Baseline = GPT-2 base (adapter en cero todavia)
acc_base = sql_accuracy()
ppl_base = perplexity(GENERAL_TEXT)
print(f"SQL accuracy (base): {acc_base:.1%}")
print(f"Perplejidad texto general (base): {ppl_base:.1f}")

[FALLO] What is the result of the game when the attendance is 62,078?
   generado: SELECT * FROM table_name_6 WHERE attendance = 62,078
   esperado: SELECT result FROM table_name_6 WHERE attendance = "62,078"



[FALLO] Which season had a home 2-3?
   generado: SELECT season FROM season WHERE season.season_id = season.season_id;
   esperado: SELECT season FROM table_name_2 WHERE home = "2-3"



[FALLO] On what Date is the match played on Clay Surface with a Score in the final of 6–2, 7–5?
   generado: SELECT * FROM table_name_32 WHERE score_in_the_final = 6;
   esperado: SELECT date FROM table_name_32 WHERE surface = "clay" AND score_in_the_final = "6–2, 7–5"



[FALLO] When Jim Thorpe of United States has a score of 71, what is the place?
   generado: The place is the name of the player.
   esperado: SELECT place FROM table_name_2 WHERE score = 71 AND country = "united states" AND player = "jim thorpe"



[FALLO] What is the custody level of the facility in Shelton?
   generado: The custody level is the level of the facility that is in the custody of the tenant.
   esperado: SELECT custody_level_s_ FROM table_25346763_1 WHERE location = "Shelton"



[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


SQL accuracy (base): 0.0%
Perplejidad texto general (base): 44.3


---

# 4 · Entrenamos · solo el adapter

Mismos hiperparámetros que el Lab 1. La única diferencia: los gradientes solo tocan el adapter,
el GPT-2 original ni se inmuta.


In [7]:
args = TrainingArguments(
    output_dir="gpt2-sql-lora",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    learning_rate=3e-4,           # LoRA suele querer un LR mas alto que el full FT (5e-5 en el Lab 1)
    fp16=(device == "cuda"),
    logging_steps=50,
    save_strategy="no",
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train_tok, data_collator=collator)
trainer.train()

Step,Training Loss
50,3.069240
100,2.023820
150,1.643403
200,1.527538
250,1.446249
300,1.414350
350,1.370064
400,1.365031
450,1.381954
500,1.357337


TrainOutput(global_step=2000, training_loss=1.3500496673583984, metrics={'train_runtime': 177.7285, 'train_samples_per_second': 90.025, 'train_steps_per_second': 11.253, 'total_flos': 946506108100608.0, 'train_loss': 1.3500496673583984, 'epoch': 2.0})

---

# 5 · ¿Sabe SQL? · casi la misma accuracy, una fracción de los pesos


In [8]:
demo = [
    ("How many employees are in the sales department?",
     "CREATE TABLE employees (name VARCHAR, salary INT, department VARCHAR)"),
    ("What is the average salary?",
     "CREATE TABLE employees (name VARCHAR, salary INT, department VARCHAR)"),
]
for q, ctx in demo:
    print("Q:", q)
    print("SQL:", generate(build_prompt(q, ctx)), "\n")

acc_lora = sql_accuracy()   # imprime ejemplos: los SQL son buenos aunque no identicos
print(f"\nMatch exacto · base: {acc_base:.1%}  ->  LoRA: {acc_lora:.1%}")
print("El match exacto es estricto (penaliza SQL valido pero distinto). Mira los SQL de arriba.")
print("Lo importante: se acerca al full FT del Lab 1 (~62% en la ejecución de referencia) entrenando <1% de los pesos.")

Q: How many employees are in the sales department?


SQL: SELECT MAX(name) FROM employees WHERE salary = "Sales" 

Q: What is the average salary?


SQL: SELECT AVG(name) FROM employees WHERE salary = "Average salary" 



[FALLO] What is the result of the game when the attendance is 62,078?
   generado: SELECT result FROM table_name_6 WHERE attendance = 62,078
   esperado: SELECT result FROM table_name_6 WHERE attendance = "62,078"



[OK ] Which season had a home 2-3?
   generado: SELECT season FROM table_name_2 WHERE home = "2-3"
   esperado: SELECT season FROM table_name_2 WHERE home = "2-3"



[FALLO] On what Date is the match played on Clay Surface with a Score in the final of 6–2, 7–5?
   generado: SELECT date FROM table_name_32 WHERE surface = "clay" AND score_in_the_final = 6–2
   esperado: SELECT date FROM table_name_32 WHERE surface = "clay" AND score_in_the_final = "6–2, 7–5"



[FALLO] When Jim Thorpe of United States has a score of 71, what is the place?
   generado: SELECT place FROM table_name_2 WHERE player = "Jim Thorpe" AND score = 71
   esperado: SELECT place FROM table_name_2 WHERE score = 71 AND country = "united states" AND player = "jim thorpe"



[OK ] What is the custody level of the facility in Shelton?
   generado: SELECT custody_level_s_ FROM table_25346763_1 WHERE location = "shelton"
   esperado: SELECT custody_level_s_ FROM table_25346763_1 WHERE location = "Shelton"




Match exacto · base: 0.0%  ->  LoRA: 30.8%
El match exacto es estricto (penaliza SQL valido pero distinto). Mira los SQL de arriba.
Lo importante: misma calidad que el full FT del Lab 1, pero entrenando <1% de los pesos.


---

# 6 · La cura · el modelo NO ha olvidado nada

Aquí esta la diferencia con el Lab 1. Como el GPT-2 original esta congelado, basta **quitar el adapter**
para recuperar al modelo base tal cual. `peft` lo hace con `disable_adapter()`.


In [9]:
print("=== Con adapter (sabe SQL) ===")
print(general_completion("The capital of France is"), "\n")

print("=== Adapter DESACTIVADO (vuelve a ser GPT-2 base) ===")
with model.disable_adapter():
    for p in GENERAL_PROMPTS:
        print(general_completion(p))
    ppl_off = perplexity(GENERAL_TEXT)

print(f"\nPerplejidad texto general · base: {ppl_base:.1f}  ->  adapter OFF: {ppl_off:.1f}")
print("Identicas: el modelo base esta intacto. NO hay olvido catastrofico.")

=== Con adapter (sabe SQL) ===


The capital of France is the capital of the French Republic, and the capital of the Republic of the United States.
The capital of the United States is the capital of the 

=== Adapter DESACTIVADO (vuelve a ser GPT-2 base) ===


Once upon a time, in a small village, a young man named Tzal, who had been a member of the village's council, was killed by a group of bandits. He was taken


The capital of France is the capital of the French Republic, and the capital of the French Republic is the capital of the French Republic.

The French Republic is the capital


To make a good cup of coffee, you need to make sure you have a good cup of coffee.

The best way to make a good cup of coffee is to make a cup of coffee

Perplejidad texto general · base: 44.3  ->  adapter OFF: 44.6
Identicas: el modelo base esta intacto. NO hay olvido catastrofico.


---

# 7 · El A/B contra el Lab 1


In [10]:
# Tamano del adapter vs el modelo completo
model.save_pretrained("gpt2-sql-lora-adapter")
adapter_path = "gpt2-sql-lora-adapter/adapter_model.safetensors"
adapter_mb = os.path.getsize(adapter_path) / 1e6 if os.path.exists(adapter_path) else None

print("                   Full FT (Lab 1)      LoRA (Lab 2)")
print("Pesos entrenados   124M (100%)          <1% (solo el adapter)")
print(f"SQL accuracy       ~62% (ref. Lab 1)    {acc_lora:.0%}")
print("Texto general      OLVIDADO             intacto (quitas el adapter)")
print(f"Guardar            ~500 MB (modelo)     ~{adapter_mb:.1f} MB (adapter)" if adapter_mb else
      "Guardar            ~500 MB (modelo)     unos pocos MB (adapter)")

                   Full FT (Lab 1)      LoRA (Lab 2)
Pesos entrenados   124M (100%)          <1% (solo el adapter)
SQL accuracy       similar              31%
Texto general      OLVIDADO             intacto (quitas el adapter)
Guardar            ~500 MB (modelo)     ~1.2 MB (adapter)


La conclusión del A/B, con la MISMA tarea y la MISMA evaluación: **accuracy de SQL cercana (unos 10 puntos por debajo), pero LoRA
no rompe el modelo y entrena 100x menos pesos.** Por eso es lo que se usa hoy.


---

# Cierre · hacia el Lab 3

Acabas de ver por que LoRA gano: adapters minusculos, base intacto, intercambiables. En el **Lab 3**
lo llevamos al stack moderno: **QLoRA + Unsloth** sobre un modelo de verdad (Gemma 4) y una tarea de
empresa medible: clasificar mensajes de soporte bancario en 77 intenciones (banking77), con el
antes/después en accuracy. El adapter, igual que aquí, se pone y se quita.
